In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess
from scipy.signal import find_peaks

In [ ]:
from scipy.stats import binom, norm

MOSC_PATH = "mosc_data.csv"
USGS_PATH = "usgs_data/usgs_m4_2005_2025.csv"

def load_moscow():
    df = pd.read_csv(MOSC_PATH, parse_dates=["datetime"])
    return df.set_index("datetime").sort_index()["value"]

def load_earthquakes(min_mag=4.0):
    df = pd.read_csv(USGS_PATH, usecols=["time", "mag"])
    df["time"] = pd.to_datetime(df["time"], utc=True).dt.tz_localize(None)
    df = df[df["mag"] >= min_mag]
    return df.set_index("time")["mag"].sort_index()

cr = load_moscow()
eq = load_earthquakes(min_mag=4.0)
print(f"CR: {len(cr)} pomiarów, {cr.index.min()} .. {cr.index.max()}")
print(f"EQ (M>=4.0): {len(eq)} zdarzeń, {eq.index.min()} .. {eq.index.max()}")

In [ ]:
def cosmoseismic_stat(cr, eq, t0, P_days, d_days, m, dt_days):
    """Test kosmo-sejsmiczny wg wzorów (1)-(4) z Homola et al. (2023)."""
    N = int(P_days // d_days)
    edges = pd.date_range(t0, periods=N + 1, freq=pd.Timedelta(days=d_days))
    eq_edges = edges + pd.Timedelta(days=dt_days)

    cr_cats = pd.cut(cr.index, edges, right=False)
    cr_binned = cr.groupby(cr_cats, observed=False).mean().reindex(cr_cats.categories)
    cr_vals = cr_binned.to_numpy()  # N binów CR

    eq_in_range = eq[(eq.index >= eq_edges[0]) & (eq.index < eq_edges[-1])]
    eq_cats = pd.cut(eq_in_range.index, eq_edges, right=False)
    eq_binned = eq_in_range.groupby(eq_cats, observed=False).sum().reindex(eq_cats.categories, fill_value=0.0)
    sm_vals = eq_binned.to_numpy()  # N binów EQ (Sm), przesuniętych o dt względem binów CR

    nCR_i, nCR_im1 = cr_vals[1:], cr_vals[:-1]
    dCR = nCR_i - nCR_im1          # i = 1..N-1 (porównania kolejnych binów)
    Sm = sm_vals[1:]               # ten sam indeks i co dCR

    med_Sm = np.nanmedian(Sm)
    med_dCR = np.nanmedian(np.abs(dCR))

    A = Sm / med_Sm - 1
    B = np.abs(dCR) / med_dCR - 1

    valid = (
        (A != 0) & (B != 0) &
        (nCR_i > 0) & (nCR_im1 > 0) &
        (Sm > 0) &
        ~np.isnan(A) & ~np.isnan(B)
    )

    c_valid = (A * B)[valid]
    Np, Nm = int((c_valid > 0).sum()), int((c_valid < 0).sum())
    n_total = Np + Nm

    if n_total == 0:
        return dict(N=N, N_valid=0, Np=0, Nm=0, PPDF=np.nan, PCDF=np.nan, sigma=np.nan)

    ppdf = binom.pmf(Np, n_total, 0.5)
    pcdf = binom.sf(Np - 1, n_total, 0.5)   # P(N+ >= obserwowane), test jednostronny
    sigma = norm.isf(pcdf)                   # odpowiednik "sigma" z artykułu

    return dict(N=N, N_valid=n_total, Np=Np, Nm=Nm, PPDF=ppdf, PCDF=pcdf, sigma=sigma)


# Walidacja: konfiguracja z artykułu dla Moskwy (oczekiwane: N+~218, N-~113, PCDF~4.1e-9, ~6 sigma)
t0 = pd.Timestamp("2013-11-14 07:00:00")
validation = cosmoseismic_stat(cr, eq, t0, P_days=1675, d_days=5, m=4.0, dt_days=15)
print("Walidacja (d=5, jak w artykule):", validation)

In [ ]:
# Pętla po szerokości binu d = 1..30 dni (Moskwa, pozostałe parametry jak wyżej)
rows = []
for d in range(1, 31):
    r = cosmoseismic_stat(cr, eq, t0, P_days=1675, d_days=d, m=4.0, dt_days=15)
    r["d"] = d
    rows.append(r)

scan = pd.DataFrame(rows).set_index("d")
scan

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

axes[0].plot(scan.index, scan["PCDF"], "o-", color="steelblue")
axes[0].set_yscale("log")
axes[0].axhline(2.87e-7, color="red", ls="--", label="5σ (jednostronnie)")
axes[0].axvline(5, color="black", ls=":", label="d=5 (wartość z artykułu)")
axes[0].set_ylabel("PCDF (im mniejsze, tym istotniejsze)")
axes[0].legend()
axes[0].grid(ls=":")

axes[1].plot(scan.index, scan["Np"], "o-", label="N+", color="green")
axes[1].plot(scan.index, scan["Nm"], "o-", label="N-", color="orange")
axes[1].axvline(5, color="black", ls=":")
axes[1].set_xlabel("Szerokość binu d [dni]")
axes[1].set_ylabel("Liczba interwałów")
axes[1].legend()
axes[1].grid(ls=":")

plt.tight_layout()
plt.show()

In [ ]:
def full_d_scan(cr, eq, t0, P_days, m, dt_days, d_range):
    return {d: cosmoseismic_stat(cr, eq, t0, P_days, d, m, dt_days)["PCDF"] for d in d_range}


def circular_shift_eq(eq, rng):
    start, end = eq.index.min(), eq.index.max()
    span = end - start
    shift = pd.Timedelta(seconds=int(rng.uniform(0, span.total_seconds())))
    new_idx = (start + ((eq.index - start + shift) % span)).astype(eq.index.dtype)
    return pd.Series(eq.values, index=new_idx).sort_index()


best_d = scan["PCDF"].idxmin()
best_pcdf = scan["PCDF"].min()
print(f"Obserwowane minimum: PCDF={best_pcdf:.3e} przy d={best_d}")

n_trials = len(scan)
bonferroni_p = min(1.0, best_pcdf * n_trials)
print(f"Bonferroni (n={n_trials} prób, zakłada niezależność): p_global <= {bonferroni_p:.3e}")

In [ ]:
# Monte Carlo: cykliczne przesunięcie EQ względem CR, żeby zniszczyć realny
# związek CR<->EQ zachowując wewnętrzną strukturę czasową obu szeregów.
# Dla każdej symulacji powtarzamy CAŁY skan d=1..30 i zapamiętujemy jego
# minimum PCDF - dystrybuanta tych minimów pod hipotezą zerową daje
# poprawną (globalną) p-wartość dla naszego obserwowanego wyniku.
# ~1000 symulacji x 30 wartości d ~= 7-8 minut.
n_sims = 1000
rng = np.random.default_rng(42)
mc_minima = np.empty(n_sims)
for i in range(n_sims):
    eq_shift = circular_shift_eq(eq, rng)
    sim = full_d_scan(cr, eq_shift, t0, 1675, 4.0, 15, range(1, 31))
    mc_minima[i] = np.nanmin(list(sim.values()))

p_global = (mc_minima <= best_pcdf).mean()
p_global_floor = max(p_global, 1 / n_sims)

In [ ]:
print(f"Monte Carlo (n_sims={n_sims}): p_global (empiryczne) = {p_global:.4f}")
print(f"  najbardziej ekstremalne minimum spod H0: {mc_minima.min():.3e}")
print(f"  mediana minimów spod H0: {np.median(mc_minima):.3e}")
print(f"  floor (bo 0/1000): p_global <= {p_global_floor:.4f} (~{norm.isf(p_global_floor):.2f} sigma)")

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(np.log10(mc_minima), bins=30, color="steelblue", edgecolor="black", alpha=0.8,
         label=f"min(PCDF) pod H0 (n={n_sims} symulacji)")
plt.axvline(np.log10(best_pcdf), color="red", ls="--",
            label=f"obserwowane min(PCDF)={best_pcdf:.1e} (d={best_d})")
plt.xlabel("log10(min PCDF w skanie d=1..30)")
plt.ylabel("liczba symulacji")
plt.legend()
plt.grid(ls=":")
plt.tight_layout()
plt.show()